# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shashank007-ux/Week-1-Run-the-Starter-Notebooks/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


## 1. My lane as an ML task (type)

I would frame this as a **ranking/scoring problem**. The decision is not just “is this page declining?” but “which pages should an editor review first?” That makes a priority score or ranked review queue the right output, because the action is to sort a list and spend limited time on the best candidates first.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from pathlib import Path
import pandas as pd


def load_refresh_data():
    base = Path.cwd()
    for candidate in [base, base.parent, base.parent.parent]:
        path = candidate / "data" / "raw" / "content_refresh_anonymized.csv"
        if path.exists():
            return pd.read_csv(path)
    raise FileNotFoundError("Could not find the starter dataset.")


df = load_refresh_data()
df.shape

(30000, 44)

## 2. Target or proxy

let's I choose the target of around 30 days. The target I would use is a proxy for future content decline: whether a page’s impressions fall in the next 30 days compared with the previous 30 days. I would use that because it is an observed outcome in the data, while the real business outcome of “refresh success” is not directly recorded. This makes the label practical and honest, even if it is a proxy rather than the final business result.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

df["declined_in_next_window"] = (
    df["impressions_last_30d"].fillna(0) < df["impressions_prev_30d"].fillna(0)
)
df[["content_id", "impressions_prev_30d", "impressions_last_30d", "declined_in_next_window"]].head(10)


,content_id,impressions_prev_30d,impressions_last_30d,declined_in_next_window
0,content_304f48230142,987,578,True
1,content_a1fb4e703a9e,5915,2501,True
2,content_9aa793d4d895,6089,2382,True
3,content_331d6c4de07b,4206,3626,True
4,content_d99b7a2d90ca,6452,4211,True
5,content_d4084a4bc775,1009,617,True
6,content_9a34b442b552,13,1,True
7,content_a63219c6e95a,632,636,False
8,content_5e6c160719bc,13828,5696,True
9,content_c27558df2b0c,356,252,True


## 3. Success metric

I would use precision@K as the main metric. In this lane, the value is not to perfectly predict every declining page, but to put the most promising review candidates near the top of the queue. A good model would rank a high share of genuinely worthwhile refresh candidates in the top section of the list, because that is the part an editor would actually review first.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Share declined:", round(df["declined_in_next_window"].mean(), 3))

Share declined: 0.657


## 4. The unit of analysis, as a real dataframe

The unit of analysis is one content item, meaning one row in the dataframe corresponds to one page or content asset that could be reviewed for refresh. The model would score each content item using its recent performance and freshness signals, and the editor would act on the ranked list of those items.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

unit_df = df[["content_id", "client_id", "content_type", "ctr", "engagement_rate", "avg_position", "impressions_90d", "days_since_last_update"]].head(10)
unit_df

,content_id,client_id,content_type,ctr,engagement_rate,avg_position,impressions_90d,days_since_last_update
0,content_304f48230142,client_f369cb89fc,keyword article,0.76,5.88,10.6,3803,20
1,content_a1fb4e703a9e,client_4e07408562,keyword article,0.05,0.00,20.3,15320,25
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,0.09,0.00,36.5,12581,20
3,content_331d6c4de07b,client_19581e27de,keyword article,0.49,1.28,6.2,11751,22
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,0.13,0.00,44.0,19140,14
5,content_d4084a4bc775,client_f369cb89fc,keyword article,0.03,0.00,8.5,3970,20
6,content_9a34b442b552,client_8722616204,keyword article,0.00,0.00,7.0,20,20
7,content_a63219c6e95a,client_19581e27de,keyword article,0.06,3.57,21.2,1724,22
8,content_5e6c160719bc,client_6208ef0f77,keyword article,0.09,5.88,46.0,32574,20
9,content_c27558df2b0c,client_19581e27de,keyword article,0.16,0.00,4.9,1240,104


## 5. Why ML beats a fixed rule here

A fixed rule is too simple because the signals are mixed and noisy. A page might have low CTR, weak freshness, and high search demand at the same time, while another page may look similar but behave differently by client or content type. ML is useful here because it can combine many weak signals into a more nuanced ranking, even when the pattern is not easy to write as a hand-built if-statement.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

example = df[["ctr", "engagement_rate", "avg_position", "impressions_90d", "days_since_last_update"]].head(10)
example

,ctr,engagement_rate,avg_position,impressions_90d,days_since_last_update
0,0.76,5.88,10.6,3803,20
1,0.05,0.00,20.3,15320,25
2,0.09,0.00,36.5,12581,20
3,0.49,1.28,6.2,11751,22
4,0.13,0.00,44.0,19140,14
5,0.03,0.00,8.5,3970,20
6,0.00,0.00,7.0,20,20
7,0.06,3.57,21.2,1724,22
8,0.09,5.88,46.0,32574,20
9,0.16,0.00,4.9,1240,104


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.